<a href="https://colab.research.google.com/github/nattaponm/Teach_Tdating_strom_tracks_HID_CloudProperties/blob/main/notebooks/00_prepare_track_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 00: เตรียมชุดข้อมูลสำหรับวิเคราะห์เส้นทางพายุ (สำหรับผู้สอน)
### Building a common 1-km dataset: CAPPI 15/10/5 min, polar products (HID, Doppler, dual-pol), DEM and T-DaTing labels

⏱️ ใช้เวลารัน **~15–25 นาทีใน Colab** (รันครั้งเดียว ผลลัพธ์อยู่ใน `data/derived/` แล้ว นิสิตไม่ต้องรัน)

**ข้อมูลต้นทาง**
| ข้อมูล | ที่มา | ความละเอียด |
|---|---|---|
| pseudo-CAPPI 2 km (dBZ) | repo [Teach_Pysteps_KKN_summerStorm](https://github.com/nattaponm/Teach_Pysteps_KKN_summerStorm) | 1 km, 15 นาที |
| polar volume 4 มุมยก (Z, V, σv, ZDR, ΦDP, ρHV) | repo [Teach_HID_Doppler_KKN_summerStorm](https://github.com/nattaponm/Teach_HID_Doppler_KKN_summerStorm) | 500 m × ~0.7°, 15 นาที |
| DEM (SRTM/GMTED) | `data/basemap/terrain_KKN_aeqd_600m.nc` | 600 m |
| sounding 48431 00Z | `data/sounding/` | — |

**ผลลัพธ์ใน `data/derived/`** (ช่วง 05:00–17:00 UTC = 12:00–24:00 น., กริด AEQD 1 km 481 × 481)
| ไฟล์ | เนื้อหา |
|---|---|
| `KKN_20240503_cappi_15min.nc` | reflectivity ที่วัดจริง 49 ภาพ |
| `KKN_20240503_cappi_05min_linZ.nc`, `..._nearest.nc` | ภาพ 5 นาที 145 ภาพ (สังเคราะห์ 2 วิธี; ตัวแปร `observed`) |
| `KKN_20240503_polar_15min.nc` | Zc, Vr (dealiased), divergence, az. shear, EDR, KDP, ZDR, HID 4 มุมยก, hail/graupel column |
| `KKN_20240503_terrain_1km.nc` | elevation, slope, relief, dh/dx, dh/dy, TPI, beam height (0.48°) |
| `KKN_20240503_labels_<tag>.nc` | ผล T-DaTing เป็น label cube (ค่า = track_id + 1) สำหรับ 15min, 10/05min × linZ/nearest |
| `KKN_20240503_interp_test.csv` | ผลทดสอบคุณภาพการแทรกภาพ (leave-one-out) |

> ⚠️ **ไม่มีข้อมูลเรดาร์ถี่กว่า 15 นาที** ภาพ 5 และ 10 นาทีจึงเป็นภาพสังเคราะห์จาก advection interpolation
> ใช้สำหรับ "ติดตามเส้นทาง" เท่านั้น — ค่าความเข้ม HID และ Doppler ใช้เฉพาะเวลาที่วัดจริง

In [1]:
#@title ⚙️ Setup: ติดตั้งแพ็กเกจและดาวน์โหลดข้อมูล (รันก่อนเสมอ ~2–4 นาทีใน Colab)
import os, sys, warnings
warnings.filterwarnings("ignore")

REPO = "Teach_Tdating_strom_tracks_HID_CloudProperties"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip install -q pysteps==1.21.5 arm_pyart cartopy netCDF4 geopandas metpy cmweather rasterio scikit-image
    !pip install -q --no-build-isolation git+https://github.com/nasa/PyTDA.git
    if not os.path.exists(REPO):
        !git clone -q --depth 1 https://github.com/nattaponm/{REPO}.git

for ROOT in [REPO, ".", ".."]:
    if os.path.exists(os.path.join(ROOT, "utils", "track_tools.py")):
        break
ROOT = os.path.abspath(ROOT)
sys.path.insert(0, ROOT)
DATA = os.path.join(ROOT, "data")
DER = os.path.join(DATA, "derived")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from utils import track_tools as tt
from utils import mapplot as mp
from utils import kkn_tools as kt
mp.set_style()
mp.set_data_dir(DATA)
print("✅ พร้อมใช้งาน | ข้อมูลอยู่ที่:", DATA)

✅ พร้อมใช้งาน | ข้อมูลอยู่ที่: /content/Teach_Tdating_strom_tracks_HID_CloudProperties/data


In [2]:
#@title 📥 ดาวน์โหลดข้อมูลต้นทางจาก repo pysteps และ repo HID/Doppler
REPO1, REPO2 = "Teach_Pysteps_KKN_summerStorm", "Teach_HID_Doppler_KKN_summerStorm"
BASE = os.path.dirname(ROOT)
for r in (REPO1, REPO2):
    if not os.path.exists(os.path.join(BASE, r)):
        !git clone -q --depth 1 https://github.com/nattaponm/{r}.git {os.path.join(BASE, r)}
SRC1 = os.path.join(BASE, REPO1, "data")
SRC2 = os.path.join(BASE, REPO2, "data")
CAPPI_SRC = os.path.join(SRC1, "KKN_20240503", "KKN_20240503_pcappi2km_dbz.nc")
os.makedirs(DER, exist_ok=True)
from utils import polar_tools as pt
T0, T1 = "05:00", "17:00"
print("pseudo-CAPPI:", os.path.exists(CAPPI_SRC), "| polar files:", len(pt.polar_files(SRC2, T0, T1)))

pseudo-CAPPI: True | polar files: 49


## 1. CAPPI 15 นาที (ข้อมูลที่วัดจริง)

In [3]:
from utils.data_io import load_pcappi, time_index
Zall, meta_all = load_pcappi(CAPPI_SRC)
i0, i1 = time_index(meta_all, T0), time_index(meta_all, T1)
Z15 = Zall[i0:i1 + 1]
t15 = list(meta_all["timestamps"][i0:i1 + 1])
assert meta_all["available"][i0:i1 + 1].all(), "ช่วง 05–17 UTC ต้องไม่มีข้อมูลขาด"
tt.write_cube_like(CAPPI_SRC, Z15, t15, os.path.join(DER, "KKN_20240503_cappi_15min.nc"), 15,
                   title="KKN pseudo-CAPPI 2 km, observed 15-min, 05-17 UTC")
print(f"บันทึก CAPPI 15 นาที: {len(t15)} ภาพ ({t15[0]:%H:%M}–{t15[-1]:%H:%M} UTC)")

บันทึก CAPPI 15 นาที: 49 ภาพ (05:00–17:00 UTC)


## 2. ทดสอบคุณภาพการแทรกภาพ (leave-one-out) แล้วสร้างภาพ 5 นาที 2 วิธี
ใช้ภาพทุก 30 นาทีแทรกกลับเป็น 15 นาที แล้วเทียบกับภาพจริงที่ตัดออก → บอกได้ว่าภาพสังเคราะห์ "เชื่อได้แค่ไหน"

In [4]:
loo = tt.loo_interpolation_test(Z15, t15)
loo.to_csv(os.path.join(DER, "KKN_20240503_interp_test.csv"), index=False)
print(loo.groupby("method")[["CSI35", "CSI45", "RMSE_dB", "A45_ratio"]].median().round(3))

Pysteps configuration file found at: /usr/local/lib/python3.13/dist-packages/pysteps/pystepsrc



             CSI35  CSI45  RMSE_dB  A45_ratio
method                                       
linZ         0.184    0.0   21.353      0.131
nearest      0.160    0.0   24.682      0.463
persistence  0.073    0.0   29.228      0.931


In [5]:
for method in ("linZ", "nearest"):
    Z5, t5, obs5 = tt.advect_interpolate(Z15, t15, n_sub=3, method=method)
    tt.write_cube_like(CAPPI_SRC, Z5, t5, os.path.join(DER, f"KKN_20240503_cappi_05min_{method}.nc"), 5,
                       title=f"KKN pseudo-CAPPI 2 km, 5-min ({method} advection interpolation)", observed=obs5)
    a45 = np.nansum(Z5 >= 45, axis=(1, 2))
    print(f"{method:8s}: {len(t5)} ภาพ | พื้นที่ ≥45 dBZ มัธยฐาน: ภาพจริง {np.median(a45[obs5]):.0f} km², ภาพแทรก {np.median(a45[~obs5]):.0f} km²")

linZ    : 145 ภาพ | พื้นที่ ≥45 dBZ มัธยฐาน: ภาพจริง 33 km², ภาพแทรก 6 km²


nearest : 145 ภาพ | พื้นที่ ≥45 dBZ มัธยฐาน: ภาพจริง 33 km², ภาพแทรก 16 km²


## 3. DEM และความสูงลำคลื่นบนกริด 1 km

In [6]:
_, meta = tt.load_cube(os.path.join(DER, "KKN_20240503_cappi_15min.nc"))
terr = tt.make_terrain_grid(meta, DATA)
terr["beam_h0_km"] = tt.beam_height_grid(meta)
tt.save_grids(os.path.join(DER, "KKN_20240503_terrain_1km.nc"), terr, meta,
              attrs={"source": "SRTM/GMTED via AWS Terrain Tiles (600 m) interpolated to 1 km", "beam_elevation_deg": 0.48})
print({k: f"{np.nanmin(v):.1f}–{np.nanmax(v):.1f}" for k, v in terr.items()})

{'elevation': '-3.2–1916.0', 'slope': '0.0–18.5', 'relief': '3.3–1571.0', 'dhdx': '-0.3–0.3', 'dhdy': '-0.3–0.3', 'tpi': '-575.4–747.5', 'beam_h0_km': '0.2–9.8'}


## 4. ผลิตภัณฑ์จาก polar ราย 15 นาที (ขั้นตอนเดียวกับ repo HID/Doppler NB2–NB5)
⏳ ~10 วินาทีต่อ volume × 49 volume

In [7]:
import gc, time
snd = pt.load_profile(os.path.join(SRC2, "sounding", "sounding_48431_2024-05-03_00Z.csv"))
stack = {k: [] for k in tt.POLAR_VARS}
ptimes = []
t_start = time.time()
for k, f in enumerate(pt.polar_files(SRC2, T0, T1)):
    r = pt.read_volume(f)
    pt.process_volume(r, snd)                         # dealias (tuned region), DIV, AZSHEAR, ΦDP/KDP, ZDR, Z-PHI, HID, EDR
    g = tt.polar_volume_to_grid(r, meta)
    for v in tt.POLAR_VARS:
        stack[v].append(g[v])
    ptimes.append(pt.volume_time(r))
    del r, g; gc.collect()
    if k % 8 == 0:
        print(f"  {ptimes[-1]:%H:%M} UTC ({time.time() - t_start:.0f} s)")
grids = {v: np.stack(stack[v]).astype("int8" if v.startswith("hid") else ("uint8" if v.endswith("_col") else "float32"))
         for v in tt.POLAR_VARS}
tt.save_grids(os.path.join(DER, "KKN_20240503_polar_15min.nc"), grids, meta, times=ptimes,
              attrs={"hid_classes": ",".join(tt.HID_NAMES), "processing": "polar_tools.process_volume (repo HID/Doppler)"})
print(f"บันทึก polar products: {len(ptimes)} เวลา ({time.time() - t_start:.0f} s)")

  05:00 UTC (6 s)


  07:00 UTC (68 s)


  09:00 UTC (157 s)


  11:00 UTC (247 s)


  13:00 UTC (336 s)


  15:00 UTC (462 s)


  17:00 UTC (530 s)


บันทึก polar products: 49 เวลา (532 s)


## 5. T-DaTing ทุกความละเอียด → label cube
พารามิเตอร์เดียวกันทุกชุด (ปรับสำหรับ KKN ใน repo pysteps NB6): `minref=33, maxref=45, mindiff=5, minsize=15, minmax=40, mindis=8, mintrack=2, match_frac=0.25`

In [8]:
import xarray as xr
sets = {"15min": os.path.join(DER, "KKN_20240503_cappi_15min.nc")}
for m in ("linZ", "nearest"):
    sets[f"05min_{m}"] = os.path.join(DER, f"KKN_20240503_cappi_05min_{m}.nc")
summary = []
for tag in tt.TAGS:
    src = sets["15min"] if tag == "15min" else sets["05min_" + tag.split("_")[1]]
    Z, mm = tt.load_cube(src)
    tl = list(mm["timestamps"])
    with xr.open_dataset(src) as ds:
        obs = ds["observed"].values.astype(bool)
    if tag.startswith("10min"):
        Z, tl, obs = Z[::2], tl[::2], obs[::2]
    t_ = time.time()
    tracks = tt.run_tdating(Z, tl)
    lab = tt.tracks_to_labels(tracks, tl, Z.shape[1:])
    ds = xr.Dataset({"label": (("time", "y", "x"), lab), "observed": ("time", obs.astype("uint8"))},
                    coords={"time": pd.to_datetime(tl), "y": mm["y"], "x": mm["x"]},
                    attrs={"projection": mm["projection"], "yorigin": "upper", "params": str(tt.PARAMS),
                           "track_kw": str(tt.TRACK_KW), "label_meaning": "track_id + 1 (0 = no cell)"})
    ds.to_netcdf(os.path.join(DER, f"KKN_20240503_labels_{tag}.nc"), encoding={"label": {"zlib": True, "complevel": 5}})
    life = np.array([(pd.Timestamp(tr["time"].iloc[-1]) - pd.Timestamp(tr["time"].iloc[0])).total_seconds() / 60 for tr in tracks])
    summary.append({"tag": tag, "frames": len(tl), "tracks": len(tracks), "life>=30min": int((life >= 30).sum()),
                    "life>=60min": int((life >= 60).sum()), "max_life_min": life.max(), "time_s": round(time.time() - t_)})
    print(summary[-1])
pd.DataFrame(summary).to_csv(os.path.join(DER, "KKN_20240503_tracking_summary.csv"), index=False)

{'tag': '15min', 'frames': 49, 'tracks': 114, 'life>=30min': 59, 'life>=60min': 12, 'max_life_min': np.float64(120.0), 'time_s': 41}


{'tag': '10min_linZ', 'frames': 73, 'tracks': 160, 'life>=30min': 62, 'life>=60min': 18, 'max_life_min': np.float64(140.0), 'time_s': 60}


{'tag': '05min_linZ', 'frames': 145, 'tracks': 208, 'life>=30min': 68, 'life>=60min': 24, 'max_life_min': np.float64(135.0), 'time_s': 124}


{'tag': '10min_nearest', 'frames': 73, 'tracks': 184, 'life>=30min': 48, 'life>=60min': 11, 'max_life_min': np.float64(120.0), 'time_s': 59}


{'tag': '05min_nearest', 'frames': 145, 'tracks': 304, 'life>=30min': 46, 'life>=60min': 8, 'max_life_min': np.float64(130.0), 'time_s': 129}


## 6. ตรวจไฟล์ผลลัพธ์

In [9]:
import json, glob
files = sorted(glob.glob(os.path.join(DER, "*")))
inv = pd.DataFrame({"file": [os.path.basename(f) for f in files], "MB": [os.path.getsize(f) / 1e6 for f in files]}).round(2)
manifest = {"case": "KKN 2024-05-03", "window_utc": [T0, T1], "grid": "AEQD 1 km 481x481, yorigin upper",
            "tdating_params": tt.PARAMS, "track_kw": tt.TRACK_KW, "main_sets": tt.MAIN,
            "note": "5/10-min frames are synthetic (advection interpolation). HID/Doppler only at observed 15-min times.",
            "sources": [REPO1, REPO2, "SRTM/GMTED terrain"], "files": inv.to_dict("records")}
json.dump(manifest, open(os.path.join(DER, "manifest.json"), "w"), indent=1, default=str)
print(f"รวม {inv['MB'].sum():.1f} MB")
inv

รวม 23.3 MB


,file,MB
0,KKN_20240503_cappi_05min_linZ.nc,3.51
1,KKN_20240503_cappi_05min_nearest.nc,3.16
2,KKN_20240503_cappi_15min.nc,1.15
3,KKN_20240503_interp_test.csv,0.01
4,KKN_20240503_labels_05min_linZ.nc,0.36
5,KKN_20240503_labels_05min_nearest.nc,0.36
6,KKN_20240503_labels_10min_linZ.nc,0.19
7,KKN_20240503_labels_10min_nearest.nc,0.19
8,KKN_20240503_labels_15min.nc,0.14
9,KKN_20240503_polar_15min.nc,9.91
